# Data Wrangling with pandas 📊

### ISA 383: Python for Business Analytics

This notebook introduces the main pandas operations for working with tables: inspect,
select, filter, combine, summarize, and reshape.


## Using this notebook in Google Colab

1. Before editing, select **File > Save a copy in Drive**.
2. Run the cells in order. Some practice cells intentionally wait for your input.
3. The notebook creates any course folders and teaching files it needs automatically.
4. Files under `/content` are temporary and disappear when the Colab runtime resets.
5. Never paste an API key into a notebook cell. Use the **Secrets** panel when instructed.

You do not need to find, copy, or type a file path for the prepared course data.


## Learning objectives 🎯

By the end of this notebook, you should be able to:

1. Create and inspect `Series` and `DataFrame` objects.
2. Read a CSV file and check its shape, columns, data types, and summary statistics.
3. Select, filter, sort, and create columns using readable pandas expressions.
4. Combine related tables with `merge()` and stack tables with `concat()`.
5. Summarize and reshape data with `agg()`, `size()`, `count()`, `transform()`,
   `pivot_table()`, and `melt()`.


## 1. Setup and a small real dataset 🌤️

We will use a fixed weather snapshot rather than a live download. That makes the output
reproducible and keeps the code focused on pandas.

The `date` column is the observation date. `station_id` is the key for the weather location.
The temperature columns are in degrees Celsius, precipitation is in millimetres, and wind
speed is in kilometres per hour.


### 🧭 Syntax first: Reading a CSV file

Read the pattern before running the example. The names in the pattern are placeholders.

```python
pd.read_csv(
    filepath,
    parse_dates=["date"],
    usecols=None,
    na_values=None,
)
```

Documentation: [read_csv documentation](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

WEATHER_CSV = "date,station_id,city,temp_max_c,temp_min_c,precipitation_mm,wind_max_kmh\n2025-01-01,DXB,Dubai,25.0,18.4,0.70,21.7\n2025-01-02,DXB,Dubai,22.6,19.9,0.00,26.1\n2025-01-03,DXB,Dubai,21.5,17.6,0.00,17.3\n2025-01-04,DXB,Dubai,22.9,13.0,0.00,14.5\n2025-01-05,DXB,Dubai,22.8,15.6,0.00,13.1\n2025-01-06,DXB,Dubai,25.1,12.5,0.00,16.5\n2025-01-07,DXB,Dubai,25.8,14.3,0.00,13.8\n2025-01-08,DXB,Dubai,24.1,14.0,0.00,9.9\n2025-01-09,DXB,Dubai,22.5,18.0,0.40,11.8\n2025-01-10,DXB,Dubai,24.8,14.2,0.00,15.0\n2025-01-11,DXB,Dubai,26.9,13.5,0.00,15.7\n2025-01-12,DXB,Dubai,24.9,13.8,0.00,10.6\n2025-01-13,DXB,Dubai,23.2,17.3,1.90,18.6\n2025-01-14,DXB,Dubai,23.5,19.6,0.10,13.6\n\n"
DATA_DIR = Path("/content/isa383/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_FILE = DATA_DIR / "uae_weather_daily.csv"
DATA_FILE.write_text(WEATHER_CSV, encoding="utf-8")

weather = pd.read_csv(DATA_FILE, parse_dates=["date"])
display(weather.head())


The examples use a frozen, fourteen-day Dubai weather extract. The values were retrieved
from the Open-Meteo Historical Weather API for 1-14 January 2025 and stored locally so
that every student runs the same file. The file is small enough to inspect directly, but
still has dates, numeric measures, a key, units, and missing-data decisions to discuss.

Source: https://archive-api.open-meteo.com/v1/archive
Location: Dubai, UAE (approximately 25.20 N, 55.30 E)


> 💭 **Quick thought question:** What does one row represent? What would change if the file contained hourly rather than daily observations?


## 2. The two core pandas objects 🧱

A `Series` is one labeled column. A `DataFrame` is a table made from aligned Series.
Start with the mental model, then use the real dataset.


### 🧭 Syntax first: Series, DataFrame, and inspection

Read the pattern before running the example. The names in the pattern are placeholders.

```python
pd.Series(data, index=None, name=None)
pd.DataFrame(data, index=None, columns=None)

df.head(n=5)
df.info()
df.describe(include=None)
```

Documentation: [Series](https://pandas.pydata.org/docs/reference/api/pandas.Series.html) · [DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)


In [2]:
example_temperatures = pd.Series(
    {"Monday": 24.0, "Tuesday": 25.5, "Wednesday": 23.0},
    name="temperature_c",
)

example_temperatures


Out[0]: 
Monday       24.0
Tuesday      25.5
Wednesday    23.0
Name: temperature_c, dtype: float64


In [3]:
print("Rows and columns:", weather.shape)
print("Column names:", weather.columns.tolist())

display(weather.info())


Rows and columns: (14, 7)
Column names: ['date', 'station_id', 'city', 'temp_max_c', 'temp_min_c', 'precipitation_mm', 'wind_max_kmh']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              14 non-null     datetime64[ns]
 1   station_id        14 non-null     object        
 2   city              14 non-null     object        
 3   temp_max_c        14 non-null     float64       
 4   temp_min_c        14 non-null     float64       
 5   precipitation_mm  14 non-null     float64       
 6   wind_max_kmh      14 non-null     float64       
dtypes: datetime64[ns](1), float64(4), object(2)
memory usage: 916.0+ bytes


None

In [4]:
weather.select_dtypes("number").describe().round(1)


Out[0]: 
       temp_max_c  temp_min_c  precipitation_mm  wind_max_kmh
count        14.0        14.0              14.0          14.0
mean         24.0        15.8               0.2          15.6
std           1.5         2.5               0.5           4.4
min          21.5        12.5               0.0           9.9
25%          22.8        13.8               0.0          13.2
50%          23.8        15.0               0.0          14.8
75%          25.0        17.9               0.1          17.1
max          26.9        19.9               1.9          26.1


### 🧩 Guided practice: Series and DataFrame basics

Create a Series called `rain_by_day` with three labeled precipitation values. Then display the first three rows of `weather` with `head()`.


In [ ]:
# TODO: Create rain_by_day, then display weather.head(3).

## 3. Reading and writing data files 💾

In practice, the data usually already exists in a file. Reading it is only the first step:
always inspect the result before analyzing it.

Common readers include `pd.read_csv()`, `pd.read_excel()`, and `pd.read_parquet()`.
The exact file format matters because it affects types, missing values, and performance.


### 🧭 Syntax first: Writing a small table

Read the pattern before running the example. The names in the pattern are placeholders.

```python
df.to_csv(filepath, index=False)
df.to_excel(filepath, index=False, sheet_name="Sheet1")
```

Documentation: [read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) · [DataFrame.to_csv](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html)


In [6]:
# `index=False` prevents pandas from writing the DataFrame index as an extra column.
weather_preview = weather.head(5)
csv_text = weather_preview.to_csv(index=False)
print(csv_text)


date,station_id,city,temp_max_c,temp_min_c,precipitation_mm,wind_max_kmh
2025-01-01,DXB,Dubai,25.0,18.4,0.7,21.7
2025-01-02,DXB,Dubai,22.6,19.9,0.0,26.1
2025-01-03,DXB,Dubai,21.5,17.6,0.0,17.3
2025-01-04,DXB,Dubai,22.9,13.0,0.0,14.5
2025-01-05,DXB,Dubai,22.8,15.6,0.0,13.1



> 💭 **Quick thought question:** Why is checking the data type of a date column important before calculating a time difference?


## 4. Selecting, filtering, and sorting 🔎

Boolean filtering creates a True/False mask. Use `&` and `|` for pandas Series, and put
each condition in parentheses. `loc` is useful when you want to select both rows and columns.


### 🧭 Syntax first: Selecting, filtering, and sorting

Read the pattern before running the example. The names in the pattern are placeholders.

```python
df.loc[row_selector, column_selector]
df.iloc[row_positions, column_positions]
df.query("column >= value")
df.sort_values(by="column", ascending=True, na_position="last")
df.assign(new_column=expression)
```

Documentation: [loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html) · [iloc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.iloc.html) · [query](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.query.html) · [sort_values](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html) · [assign](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.assign.html)


In [7]:
warm_days = weather.loc[
    weather["temp_max_c"] >= 25,
    ["date", "temp_max_c", "temp_min_c"],
]
warm_days


Out[0]: 
         date  temp_max_c  temp_min_c
0  2025-01-01        25.0        18.4
5  2025-01-06        25.1        12.5
6  2025-01-07        25.8        14.3
10 2025-01-11        26.9        13.5


In [8]:
rainy_days = weather.query("precipitation_mm > 0")
rainy_days[["date", "precipitation_mm"]]


Out[0]: 
         date  precipitation_mm
0  2025-01-01               0.7
8  2025-01-09               0.4
12 2025-01-13               1.9
13 2025-01-14               0.1


In [9]:
# Add a derived column, then sort from the largest temperature range to the smallest.
weather_with_range = weather.assign(
    temp_range_c=(weather["temp_max_c"] - weather["temp_min_c"]).round(1),
    rainy=weather["precipitation_mm"] > 0,
)

weather_with_range.sort_values("temp_range_c", ascending=False).head()


Out[0]: 
         date station_id   city  ...  wind_max_kmh  temp_range_c  rainy
10 2025-01-11        DXB  Dubai  ...          15.7          13.4  False
5  2025-01-06        DXB  Dubai  ...          16.5          12.6  False
6  2025-01-07        DXB  Dubai  ...          13.8          11.5  False
11 2025-01-12        DXB  Dubai  ...          10.6          11.1  False
9  2025-01-10        DXB  Dubai  ...          15.0          10.6  False

[5 rows x 9 columns]


> 💭 **Quick thought question:** When is `query()` easier to read than a long boolean expression? When might the longer expression be clearer?


## 5. Combining tables: `merge()` and `concat()` 🔗

`merge()` is a database-style join. It uses a key to attach columns from one table to another.
`concat()` stacks compatible tables. Use `merge()` when the tables describe related entities;
use `concat()` when the tables contain more rows of the same kind.


### 🧭 Syntax first: Combining tables

Read the pattern before running the example. The names in the pattern are placeholders.

```python
left.merge(
    right,
    how="left",
    on="key",
    validate="many_to_one",
    indicator=False,
)

pd.concat(
    [df1, df2],
    axis=0,
    ignore_index=True,
    join="outer",
)
```

Documentation: [merge](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html) · [concat](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)


In [10]:
stations = pd.DataFrame(
    {
        "station_id": ["DXB", "AUH"],
        "region": ["Dubai", "Abu Dhabi"],
        "country": ["UAE", "UAE"],
    }
)

weather_enriched = weather.merge(
    stations,
    on="station_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

weather_enriched[["station_id", "region", "date", "temp_max_c", "_merge"]].head()


Out[0]: 
  station_id region       date  temp_max_c _merge
0        DXB  Dubai 2025-01-01        25.0   both
1        DXB  Dubai 2025-01-02        22.6   both
2        DXB  Dubai 2025-01-03        21.5   both
3        DXB  Dubai 2025-01-04        22.9   both
4        DXB  Dubai 2025-01-05        22.8   both


> 💭 **Quick thought question:** What should happen if a weather row contains a station ID that is missing from the lookup table?


In [11]:
first_week = weather.iloc[:7].copy()
second_week = weather.iloc[7:].copy()
recombined = pd.concat([first_week, second_week], ignore_index=True)

print("Rows after concat:", len(recombined))
recombined.head(2)


Rows after concat: 14
Out[0]: 
        date station_id   city  ...  temp_min_c  precipitation_mm  wind_max_kmh
0 2025-01-01        DXB  Dubai  ...        18.4               0.7          21.7
1 2025-01-02        DXB  Dubai  ...        19.9               0.0          26.1

[2 rows x 7 columns]


## 6. Summarizing with `groupby()` 📈

Grouping follows a simple pattern: split the rows into groups, calculate a summary for each
group, calculate a summary for each group, and combine the results. The parameters control
the grouping key, the index, sorting, and how missing group labels are handled.


### 🧭 Syntax first: Grouping, aggregation, and transformation

Read the pattern before running the example. The names in the pattern are placeholders.

```python
grouped = df.groupby(
    by="group_column",
    as_index=True,
    sort=True,
    dropna=True,
)

grouped.agg(
    rows=("value", "size"),
    non_missing=("value", "count"),
    average=("value", "mean"),
)

grouped.size()
grouped["value"].count()
grouped["value"].transform("mean")
grouped.filter(lambda group: group["value"].mean() > 0)
```

Documentation: [groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) · [agg](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.agg.html) · [size](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.size.html) · [count](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.count.html) · [transform](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.SeriesGroupBy.transform.html) · [filter](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.filter.html)


In [12]:
# `size` counts rows. `count` counts non-missing values in a column.
groupby_demo = weather_with_range[
    ["rainy", "date", "temp_max_c", "precipitation_mm"]
].copy()
groupby_demo.loc[groupby_demo.index[0], "temp_max_c"] = pd.NA

size_by_rain = groupby_demo.groupby("rainy").size().rename("rows")
count_by_rain = (
    groupby_demo.groupby("rainy")["temp_max_c"]
    .count()
    .rename("non_missing_highs")
)

pd.concat([size_by_rain, count_by_rain], axis=1)


Out[0]: 
       rows  non_missing_highs
rainy                         
False    10                 10
True      4                  3


In [13]:
summary_by_rain = (
    weather_with_range.groupby("rainy", as_index=False, sort=True, dropna=False)
    .agg(
        days=("date", "size"),
        non_missing_highs=("temp_max_c", "count"),
        average_high_c=("temp_max_c", "mean"),
        coldest_high_c=("temp_max_c", "min"),
        hottest_high_c=("temp_max_c", "max"),
        total_rain_mm=("precipitation_mm", "sum"),
    )
    .round(1)
)

summary_by_rain


Out[0]: 
   rainy  days  ...  hottest_high_c  total_rain_mm
0  False    10  ...            26.9            0.0
1   True     4  ...            25.0            3.1

[2 rows x 7 columns]


In [14]:
# A list of functions is useful when several summaries apply to one column.
weather_with_range.groupby("rainy")["temp_max_c"].agg(
    ["count", "mean", "min", "max"]
).round(1)


Out[0]: 
       count  mean   min   max
rainy                         
False     10  24.1  21.5  26.9
True       4  23.6  22.5  25.0


In [15]:
weather_with_range["rain_group_mean_high_c"] = (
    weather_with_range.groupby("rainy")["temp_max_c"].transform("mean").round(1)
)
weather_with_range["high_minus_rain_group_mean_c"] = (
    weather_with_range["temp_max_c"] - weather_with_range["rain_group_mean_high_c"]
).round(1)

weather_with_range[
    ["date", "rainy", "temp_max_c", "rain_group_mean_high_c", "high_minus_rain_group_mean_c"]
].head()


Out[0]: 
        date  rainy  ...  rain_group_mean_high_c  high_minus_rain_group_mean_c
0 2025-01-01   True  ...                    23.6                           1.4
1 2025-01-02  False  ...                    24.1                          -1.5
2 2025-01-03  False  ...                    24.1                          -2.6
3 2025-01-04  False  ...                    24.1                          -1.2
4 2025-01-05  False  ...                    24.1                          -1.3

[5 rows x 5 columns]


In [16]:
# `filter` keeps every row in groups that meet a group-level rule.
wide_range_groups = weather_with_range.groupby("rainy").filter(
    lambda group: group["temp_range_c"].mean() > 8
)
wide_range_groups["rainy"].value_counts()


Out[0]: 
rainy
False    10
Name: count, dtype: int64


In [17]:
weather_with_range["station_average_high_c"] = (
    weather_with_range.groupby("station_id")["temp_max_c"].transform("mean").round(1)
)
weather_with_range[["date", "temp_max_c", "station_average_high_c"]].head()


Out[0]: 
        date  temp_max_c  station_average_high_c
0 2025-01-01        25.0                    24.0
1 2025-01-02        22.6                    24.0
2 2025-01-03        21.5                    24.0
3 2025-01-04        22.9                    24.0
4 2025-01-05        22.8                    24.0


> 💭 **Quick thought question:** Why does `transform()` return the same number of rows as the original DataFrame, while `agg()` usually returns one row per group?


## 7. Reshaping data: pivot and melt 🔄

A pivot table creates a compact summary for a report. `melt()` converts wide data to long
data, which is often easier to use for plotting and repeated measurements.


### 🧭 Syntax first: Pivoting and melting

Read the pattern before running the example. The names in the pattern are placeholders.

```python
df.pivot_table(
    values="value",
    index="row_group",
    columns="column_group",
    aggfunc="mean",
    fill_value=0,
    margins=False,
)

df.melt(
    id_vars=["id"],
    value_vars=["value_a", "value_b"],
    var_name="measure",
    value_name="value",
)
```

Documentation: [pivot_table](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot_table.html) · [melt](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.melt.html)


In [18]:
weather_with_range["day_name"] = weather_with_range["date"].dt.day_name()

weekday_summary = weather_with_range.pivot_table(
    index="day_name",
    values=["temp_max_c", "temp_min_c"],
    aggfunc="mean",
).round(1)

weekday_summary


Out[0]: 
           temp_max_c  temp_min_c
day_name                         
Friday           23.2        15.9
Monday           24.2        14.9
Saturday         24.9        13.2
Sunday           23.8        14.7
Thursday         22.6        19.0
Tuesday          24.6        17.0
Wednesday        24.6        16.2


In [19]:
rain_temperature_pivot = weather_with_range.pivot_table(
    values="temp_max_c",
    index="day_name",
    columns="rainy",
    aggfunc="mean",
    fill_value=0,
    margins=True,
).round(1)

rain_temperature_pivot


Out[0]: 
rainy      False  True   All
day_name                    
Friday      23.2   0.0  23.2
Monday      25.1  23.2  24.2
Saturday    24.9   0.0  24.9
Sunday      23.8   0.0  23.8
Thursday    22.6  22.5  22.6
Tuesday     25.8  23.5  24.6
Wednesday   24.1  25.0  24.6
All         24.1  23.6  24.0


In [20]:
temperature_long = weather.melt(
    id_vars=["date", "station_id"],
    value_vars=["temp_max_c", "temp_min_c"],
    var_name="measure",
    value_name="temperature_c",
)

temperature_long.head()


Out[0]: 
        date station_id     measure  temperature_c
0 2025-01-01        DXB  temp_max_c           25.0
1 2025-01-02        DXB  temp_max_c           22.6
2 2025-01-03        DXB  temp_max_c           21.5
3 2025-01-04        DXB  temp_max_c           22.9
4 2025-01-05        DXB  temp_max_c           22.8


## 8. A readable mini-pipeline 🧪

Method chaining can make a sequence of transformations easy to follow. Use it after you
understand the individual operations. During debugging, separate steps are often clearer.


### 🧭 Syntax first: Chaining common DataFrame methods

Read the pattern before running the example. The names in the pattern are placeholders.

```python
(
    df.assign(new_column=expression)
    .loc[:, ["column_a", "new_column"]]
    .sort_values("new_column", ascending=False)
    .head(5)
)
```

Documentation: [assign](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.assign.html) · [sort_values](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html) · [head](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html)


In [21]:
top_temperature_ranges = (
    weather.assign(temp_range_c=(weather["temp_max_c"] - weather["temp_min_c"]).round(1))
    .loc[:, ["date", "city", "temp_range_c"]]
    .sort_values("temp_range_c", ascending=False)
    .head(5)
)

top_temperature_ranges


Out[0]: 
         date   city  temp_range_c
10 2025-01-11  Dubai          13.4
5  2025-01-06  Dubai          12.6
6  2025-01-07  Dubai          11.5
11 2025-01-12  Dubai          11.1
9  2025-01-10  Dubai          10.6


## Comprehensive exercises 📝

Complete the exercises below. For each one, display a small result and write one sentence
explaining what the result means. Keep the code readable; a short sequence of clear steps is
better than a clever one-liner.


### Exercise 1: Inspection and interpretation

Use `shape`, `dtypes`, and a summary method to inspect `weather`. Identify the coldest day and report the relevant values.


In [ ]:
# Write your solution here.


### Exercise 2: Filtering and sorting

Find days with wind speed above 18 km/h. Return only the date, wind speed, precipitation, and maximum temperature, sorted from highest wind speed to lowest.


In [ ]:
# Write your solution here.


### Exercise 3: Enriching with a lookup table

Create a two-row lookup table with a `station_id` and a `climate_zone`. Merge it with `weather` and use `validate='many_to_one'`.


In [ ]:
# Write your solution here.


### Exercise 4: Groupby and pivot

Create a summary comparing rainy and non-rainy days. Include the number of days, average high temperature, and total precipitation. Then reshape one result into a pivot table.


In [ ]:
# Write your solution here.


### Exercise 5: A small business question

Choose one question that a delivery company, facilities team, or outdoor-events manager could ask about this weather file. Answer it with one table and at least two pandas operations. State one limitation caused by the small time window.


In [ ]:
# Write your solution here.


## Summary ✅

The core pandas workflow is:

1. Read the data and inspect its structure.
2. Select, filter, sort, and create useful columns.
3. Combine tables carefully using explicit keys and validation.
4. Summarize with `groupby()` and reshape with pivot or melt.
5. Explain what the result means and what the data cannot establish.
